In [ ]:
import os

# Vast.ai HF credential location
os.environ["HF_HOME"] = "/workspace/.hf_home"

from huggingface_hub import login, get_token

hf_token = get_token()

if hf_token:
    print("Using cached Hugging Face authentication.")
else:
    raise RuntimeError("Hugging Face token not found.")


Hugging Face token not found. Add it in Colab Secrets.


In [ ]:
# Install specific compatible versions
import sys

!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn


!{sys.executable} -m pip install git+https://github.com/AIPHES/emnlp19-moverscore.git

!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ Error: {e}")

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
  Cloning https://github.com/AIPHES/emnlp19-moverscore.git to ./tmp/pip-req-build-yisx4o1z
  Running command git clone --filter=blob:none --quiet https://github.com/AIPHES/emnlp19-moverscore.git /tmp/pip-req-build-yisx4o1z
  Resolved https://github.com/AIPHES/emnlp19-moverscore.git to commit 0459a3b3b3bd73baa0cc515a355228ee5a2887e0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparin

Failed to load /venv/main/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /venv/main/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /venv/main/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /venv/main/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MoverScore installed successfully


In [ ]:
import sys

PYTHON = sys.executable

!{PYTHON} -m pip install -q "datasets<4.0.0"

In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import os
import threading
import time
import json
from typing import Optional
from PIL import Image
from io import BytesIO
import requests


# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

import re
import numpy as np
from transformers import EvalPrediction
import nltk
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    MllamaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import LoraConfig, get_peft_model


def load_image_from_source(image_source):
    """
    Robust image loading for VQAv2 examples.

    HuggingFaceM4/VQAv2's `image` column is already a decoded PIL.Image
    (the loading script embeds the COCO images directly), so this is
    mostly a defensive convert-to-RGB helper with a couple of fallbacks
    for paths/URLs/bytes in case a different source ever gets swapped in.
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        if isinstance(image_source, str):
            if image_source.startswith(('http://', 'https://')):
                response = requests.get(image_source, timeout=10)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content))
                return image.convert('RGB') if image.mode != 'RGB' else image

            if os.path.exists(image_source):
                image = Image.open(image_source)
                return image.convert('RGB') if image.mode != 'RGB' else image

            try:
                import base64
                image_data = base64.b64decode(image_source)
                image = Image.open(BytesIO(image_data))
                return image.convert('RGB') if image.mode != 'RGB' else image
            except Exception:
                pass

        if isinstance(image_source, bytes):
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


def extract_raw_answers(example):
    """
    Return the list of raw human answer strings for a VQAv2 example, for
    use with the official VQA soft-accuracy metric.

    HuggingFaceM4/VQAv2's `answers` field is a list of dicts like
    {"answer": "net", "answer_confidence": "yes", "answer_id": 1}. This
    also tolerates a flat list-of-strings format in case the dataset
    source is ever swapped for a different VQAv2 mirror.
    """
    answers_field = example.get('answers')
    if not answers_field:
        return []
    raw = []
    for a in answers_field:
        ans = a.get('answer') if isinstance(a, dict) else a
        if ans:
            raw.append(str(ans))
    return raw

class ReproducibilityTracker:
    """
    Tracks training reproducibility / efficiency metrics.

    Step latency is measured from on_step_begin to on_step_end,
    corresponding to one optimizer step including gradient accumulation.
    """

    def __init__(self):
        self.step_times = []
        self.training_start_time = None
        self.training_end_time = None
        self._step_start_time = None
        self.peak_memory_allocated = 0.0
        self.peak_memory_reserved = 0.0

    def start(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        self.training_start_time = time.perf_counter()

    def start_step(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self._step_start_time = time.perf_counter()

    def end_step(self):
        if self._step_start_time is None:
            return

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - self._step_start_time

        if elapsed > 0:
            self.step_times.append(elapsed)

        self._step_start_time = None

    def stop(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.training_end_time = time.perf_counter()

        if torch.cuda.is_available():
            self.peak_memory_allocated = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            )
            self.peak_memory_reserved = (
                torch.cuda.max_memory_reserved() / (1024 ** 3)
            )

    def summary(self):
        if (
            self.training_start_time is None
            or self.training_end_time is None
        ):
            return {}

        total_time = (
            self.training_end_time
            - self.training_start_time
        )

        if self.step_times:
            # Exclude first step because of CUDA/kernel initialization.
            measured_steps = (
                self.step_times[1:]
                if len(self.step_times) > 1
                else self.step_times
            )

            mean_ms = float(np.mean(measured_steps)) * 1000
            p99_ms = float(
                np.percentile(measured_steps, 99)
            ) * 1000
        else:
            mean_ms = 0.0
            p99_ms = 0.0

        return {
            "training_time_sec": total_time,
            "training_time_min": total_time / 60,
            "training_time_hours": total_time / 3600,
            "num_optimizer_steps": len(self.step_times),
            "mean_step_ms": mean_ms,
            "p99_step_ms": p99_ms,
            "peak_gpu_memory_allocated_gb": self.peak_memory_allocated,
            "peak_gpu_memory_reserved_gb": self.peak_memory_reserved,
        }


class ReproducibilityCallback(TrainerCallback):
    def __init__(self, tracker):
        self.tracker = tracker

    def on_train_begin(self, args, state, control, **kwargs):
        self.tracker.start()

    def on_step_begin(self, args, state, control, **kwargs):
        self.tracker.start_step()

    def on_step_end(self, args, state, control, **kwargs):
        self.tracker.end_step()

    def on_train_end(self, args, state, control, **kwargs):
        self.tracker.stop()

class LoRAConfig:
    """Configuration for LoRA-only training on Llama 3.2 11B Vision Instruct models."""

    def __init__(self):
        self.model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
        self.batch_size = 1
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3
        self.learning_rate = 1e-5
        self.precision = "bf16"
        self.max_length = 1024

        # LoRA configuration
        self.lora_rank = 8
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Llama Vision target modules
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Vision encoder LoRA
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        self.dataset_name = "HuggingFaceM4/VQAv2"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.answer_max_length = 512
        self.ignore_index = -100

        # Since VQAv2's real "test" split has no answers, we hold out a
        # slice of "validation" to use as our own answer-bearing test set.
        self.held_out_test_size = 1000


def main():
    config = LoRAConfig()

    print("Loading processor and model for Lora Llama Vision...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token



    model = MllamaForConditionalGeneration.from_pretrained(
        config.model_id,
        token=hf_token,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )

    # Apply LoRA configuration
    # print("=" * 80)
    # print("TOKENIZER DEBUG")
    # print("=" * 80)

    # print("Tokenizer:", type(processor.tokenizer))
    # print("Vocab size:", len(processor.tokenizer))

    # print("EOS token:", repr(processor.tokenizer.eos_token))
    # print("EOS ID:", processor.tokenizer.eos_token_id)

    # print("PAD token:", repr(processor.tokenizer.pad_token))
    # print("PAD ID:", processor.tokenizer.pad_token_id)

    # print("UNK token:", repr(processor.tokenizer.unk_token))
    # print("UNK ID:", processor.tokenizer.unk_token_id)

    # # Inspect the suspicious character
    # bad_char = "锦"

    # bad_ids = processor.tokenizer.encode(
    #     bad_char,
    #     add_special_tokens=False
    # )

    # print(f"\nTokenization of {bad_char!r}:")
    # print("IDs:", bad_ids)

    # for tid in bad_ids:
    #     print(
    #         f"  ID {tid}: "
    #         f"token={processor.tokenizer.convert_ids_to_tokens(tid)!r}, "
    #         f"decoded={processor.tokenizer.decode([tid])!r}"
    #     )
    target_modules = config.lora_target_modules.copy()

    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    meta_params = [
        name for name, param in model.named_parameters()
        if param.device.type == "meta"
    ]

    if meta_params:
        raise RuntimeError(
            f"Model still contains {len(meta_params)} meta parameters. "
            f"First few: {meta_params[:10]}"
        )

    print("✓ Model has no meta parameters")
    print(f"✓ Model device: {next(model.parameters()).device}")

    # Count vision and language modules
    from peft.tuners.lora import LoraLayer
    vision_modules = 0
    language_modules = 0

    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0:
            name_lower = name.lower()
            is_vision = any([
                'vision_tower' in name_lower,
                'vision_model' in name_lower,
                'vision_encoder' in name_lower,
                'visual' in name_lower and 'language' not in name_lower,
                name.startswith('model.vision'),
                name.startswith('vision_tower'),
            ])

            if is_vision:
                vision_modules += 1
            else:
                language_modules += 1

    print(f"Instrumented {vision_modules} vision modules + {language_modules} language modules")
    print(f"Total LoRA modules: {vision_modules + language_modules}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False

    from datasets import load_dataset, Dataset

    print("Streaming VQAv2 dataset (no full download)...")

    streamed = load_dataset(config.dataset_name, trust_remote_code=True, streaming=True)

    train_stream = streamed["train"].shuffle(seed=42, buffer_size=1000)
    train_examples = list(train_stream.take(8000))


    val_stream = streamed["validation"].shuffle(seed=42, buffer_size=1000)
    val_examples = list(val_stream.take(1000 + config.held_out_test_size))

    train_dataset = Dataset.from_list(train_examples)
    val_dataset = Dataset.from_list(val_examples[:1000])
    test_dataset = Dataset.from_list(val_examples[1000:])

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    def tokenize_vqa(example):
        """Process VQAv2 examples with robust image loading for Llama Vision."""
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = load_image_from_source(example['image'])

            if image is None:
                return None

            if not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            question = (example.get('question') or '').strip()
            question_tokens = processor.tokenizer.encode(
                question,
                add_special_tokens=False
            )

            question_tokens = question_tokens[:80]

            question = processor.tokenizer.decode(
                question_tokens,
                skip_special_tokens=True
            )

            answer = (example.get('multiple_choice_answer') or '').strip()
            answer_tokens_full = processor.tokenizer.encode(
                answer,
                add_special_tokens=False
            )

            answer_tokens_full = answer_tokens_full[:120]

            answer = processor.tokenizer.decode(
                answer_tokens_full,
                skip_special_tokens=True
            )

            if not question or not answer:
                return None

            # VQAv2 has no `context` field, so the prompt is just the question.
            text_content = f"Question: {question}\n\nProvide a concise answer."

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": text_content}
                    ],
                }
            ]

            text_prompt = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False,
            )

            inputs = processor(
                text=text_prompt,
                images=image,
                return_tensors="pt",
                add_special_tokens=False,
                truncation=True,
                max_length=config.max_length,
            )



            # Tokenize answer separately
            # Tokenize answer separately
            answer_tokens = processor.tokenizer(
                answer,
                add_special_tokens=False
            )['input_ids']

            answer_tokens = torch.tensor(
                answer_tokens,
                dtype=torch.long
            )

            # Prepare prompt tensors
            input_ids = inputs['input_ids'].squeeze(0)
            attention_mask = inputs['attention_mask'].squeeze(0)

            if not isinstance(input_ids, torch.Tensor):
                input_ids = torch.tensor(
                    input_ids,
                    dtype=torch.long
                )

            if not isinstance(attention_mask, torch.Tensor):
                attention_mask = torch.tensor(
                    attention_mask,
                    dtype=torch.long
                )

            # Create labels for the existing prompt sequence.
            # The answer is supervised at the final answer_len positions.
            labels = torch.full_like(
                input_ids,
                config.ignore_index
            )

            labels[input_ids == processor.tokenizer.pad_token_id] = config.ignore_index

            answer_len = min(
                answer_tokens.shape[0],
                input_ids.shape[0]
            )

            if answer_len > 0:
                labels[-answer_len:] = answer_tokens[-answer_len:]

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
                'vqa_answers': extract_raw_answers(example),
            }


            if 'pixel_values' in inputs:
                pixel_values = inputs['pixel_values']

                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() >= 2 and pixel_values.shape[0] == 1:
                        pixel_values = pixel_values.squeeze(0)

                    output['pixel_values'] = pixel_values

                elif isinstance(pixel_values, list) and len(pixel_values) > 0:
                    if isinstance(pixel_values[0], torch.Tensor):
                        output['pixel_values'] = (
                            pixel_values[0]
                            if len(pixel_values) == 1
                            else torch.cat(pixel_values, dim=0)
                        )

            if 'aspect_ratio_ids' in inputs:
                output['aspect_ratio_ids'] = inputs['aspect_ratio_ids']

            if 'aspect_ratio_mask' in inputs:
                output['aspect_ratio_mask'] = inputs['aspect_ratio_mask']

            if 'cross_attention_mask' in inputs:
                output['cross_attention_mask'] = inputs['cross_attention_mask']


            for key in ['input_ids', 'attention_mask', 'labels']:
                if key in output and not isinstance(output[key], torch.Tensor):
                    print(f"ERROR: {key} is not a tensor: {type(output[key])}")
                    return None

            if not hasattr(tokenize_vqa, "_printed"):
                print("\n==== SHAPES ====")
                for k, v in output.items():
                    if isinstance(v, torch.Tensor):
                        print(k, v.shape, v.dtype)
                    else:
                        print(k, type(v))
                tokenize_vqa._printed = True

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Tokenizing datasets...")
    print("Processing training data...")
    tokenized_train_dataset = train_dataset.map(
        tokenize_vqa,
        remove_columns=train_dataset.column_names,
        num_proc=None,
        desc="Processing training data",
        load_from_cache_file=False,
        writer_batch_size=1
    )

    print("Filtering valid training samples...")
    tokenized_train_dataset = tokenized_train_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering training data"
    )

    print("Processing validation data...")
    tokenized_val_dataset = val_dataset.map(
        tokenize_vqa,
        remove_columns=val_dataset.column_names,
        num_proc=None,
        desc="Processing validation data",
        load_from_cache_file=False,
        writer_batch_size=1
    )

    print("Filtering valid validation samples...")
    tokenized_val_dataset = tokenized_val_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering validation data"
    )

    print("Processing test data (held out from validation)...")
    tokenized_test_dataset = test_dataset.map(
        tokenize_vqa,
        remove_columns=test_dataset.column_names,
        num_proc=None,
        desc="Tokenizing test",
        load_from_cache_file=False,
        writer_batch_size=1
    )

    print("Filtering valid test samples...")
    tokenized_test_dataset = tokenized_test_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering test data"
    )

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_vqa_lora_llama",
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),
        max_grad_norm=0.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=config.num_workers,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,
        dataloader_pin_memory=False,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        save_total_limit=2,
        logging_steps=50,
        warmup_steps=200,
        optim="adamw_torch",
        #load_best_model_at_end=True,
        #metric_for_best_model="loss",
    )

    def collate_fn(batch):
        """Custom collator for Llama Vision inputs with proper batching."""
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]

        if len(batch) == 0:
            print("WARNING: Empty batch after filtering - creating dummy batch")
            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_pixel_values = torch.zeros((1, 1, 4, 3, 560, 560))
            dummy_aspect_ratio_ids = torch.tensor([[2]])
            dummy_aspect_ratio_mask = torch.tensor([[[1, 1, 0, 0]]])
            dummy_cross_attention_mask = torch.tensor([[[[1, 1, 0, 0]]]])

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'pixel_values': dummy_pixel_values,
                'aspect_ratio_ids': dummy_aspect_ratio_ids,
                'aspect_ratio_mask': dummy_aspect_ratio_mask,
                'cross_attention_mask': dummy_cross_attention_mask,
                'labels': dummy_labels,
            }

        try:
            # Convert non-tensor items to tensors
            for i, item in enumerate(batch):
                for key in ['input_ids', 'attention_mask', 'labels']:
                    if key in item and not isinstance(item[key], torch.Tensor):
                        batch[i][key] = torch.tensor(item[key], dtype=torch.long)

            # Pad sequences
            input_ids = torch.nn.utils.rnn.pad_sequence(
                [b['input_ids'] for b in batch],
                batch_first=True,
                padding_value=processor.tokenizer.pad_token_id
            )

            attention_mask = torch.nn.utils.rnn.pad_sequence(
                [b['attention_mask'] for b in batch],
                batch_first=True,
                padding_value=0
            )


            labels = torch.nn.utils.rnn.pad_sequence(
                [b['labels'] for b in batch],
                batch_first=True,
                padding_value=config.ignore_index
            )

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            }

            # Handle pixel_values for Llama Vision
            if 'pixel_values' in batch[0] and batch[0]['pixel_values'] is not None:
                pixel_values_list = []

                for b in batch:
                    pv = b['pixel_values']

                    if isinstance(pv, list):
                        if len(pv) > 0 and isinstance(pv[0], torch.Tensor):
                            pv = torch.stack(pv) if len(pv) > 1 else pv[0]
                        elif len(pv) > 0 and isinstance(pv[0], list):
                            pv = torch.tensor(pv)
                        else:
                            continue

                    if isinstance(pv, torch.Tensor):
                        # Ensure batch dimension is removed for individual samples
                        if pv.dim() == 7 and pv.shape[0] == 1:
                            pv = pv.squeeze(0)
                        pixel_values_list.append(pv)

                if len(pixel_values_list) > 0:
                    try:
                        # Stack along new batch dimension
                        output['pixel_values'] = torch.stack(pixel_values_list, dim=0)
                    except RuntimeError:
                        # If shapes don't match, pad to max shape
                        max_shapes = [max(pv.shape[i] for pv in pixel_values_list) for i in range(pixel_values_list[0].dim())]
                        padded_pvs = []
                        for pv in pixel_values_list:
                            if list(pv.shape) != max_shapes:
                                padding = []
                                for i in range(len(max_shapes) - 1, -1, -1):
                                    padding.extend([0, max_shapes[i] - pv.shape[i]])
                                pv = torch.nn.functional.pad(pv, padding)
                            padded_pvs.append(pv)
                        output['pixel_values'] = torch.stack(padded_pvs, dim=0)

                if 'aspect_ratio_ids' in batch[0]:
                    output['aspect_ratio_ids'] = torch.stack([
                        torch.tensor(b['aspect_ratio_ids'])
                        if not isinstance(b['aspect_ratio_ids'], torch.Tensor)
                        else b['aspect_ratio_ids']
                        for b in batch
                    ], dim=0).squeeze(1)

                if 'aspect_ratio_mask' in batch[0]:
                    output['aspect_ratio_mask'] = torch.stack([
                        torch.tensor(b['aspect_ratio_mask'])
                        if not isinstance(b['aspect_ratio_mask'], torch.Tensor)
                        else b['aspect_ratio_mask']
                        for b in batch
                    ], dim=0).squeeze(1)

                if 'cross_attention_mask' in batch[0]:
                    output['cross_attention_mask'] = torch.stack([
                        torch.tensor(b['cross_attention_mask'])
                        if not isinstance(b['cross_attention_mask'], torch.Tensor)
                        else b['cross_attention_mask']
                        for b in batch
                    ], dim=0).squeeze(1)

                output['vqa_answers'] = [b.get('vqa_answers', []) for b in batch]

                return output

        except Exception as e:
            print(f"ERROR in collate_fn: {e}")
            import traceback
            traceback.print_exc()

            # Return minimal dummy batch
            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_pixel_values = torch.zeros((1, 1, 4, 3, 560, 560))
            dummy_aspect_ratio_ids = torch.tensor([[2]])
            dummy_aspect_ratio_mask = torch.tensor([[[1, 1, 0, 0]]])
            dummy_cross_attention_mask = torch.tensor([[[[1, 1, 0, 0]]]])

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'pixel_values': dummy_pixel_values,
                'aspect_ratio_ids': dummy_aspect_ratio_ids,
                'aspect_ratio_mask': dummy_aspect_ratio_mask,
                'cross_attention_mask': dummy_cross_attention_mask,
                'labels': dummy_labels,
            }

    def make_compute_metrics(processor, eval_answers_buffer):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True);
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


        def extract_answer(text):
            """Clean Llama Vision output safely without truncating content."""

            if text is None:
                return ""

            # Remove special tokens
            text = text.replace("<bos>", "")
            text = text.replace("<eos>", "")
            text = text.replace("<start_of_turn>", "")
            text = text.replace("<end_of_turn>", "")

            # Remove leading 'model' tag if present
            text = re.sub(r"^\s*model\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = text.replace("锦", " ")
            text = re.sub(r"\s+", " ", text).strip()

            return text

        _vqa_contractions = {
            "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've",
            "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
            "hadnt": "hadn't", "hasnt": "hasn't", "havent": "haven't", "hed": "he'd",
            "hes": "he's", "im": "i'm", "ive": "i've", "isnt": "isn't", "its": "it's",
            "lets": "let's", "shant": "shan't", "shes": "she's", "shouldve": "should've",
            "shouldnt": "shouldn't", "thats": "that's", "theres": "there's", "theyd": "they'd",
            "theyre": "they're", "theyve": "they've", "wasnt": "wasn't", "wed": "we'd",
            "were": "we're", "weve": "we've", "werent": "weren't", "whats": "what's",
            "wheres": "where's", "whos": "who's", "wont": "won't", "wouldve": "would've",
            "wouldnt": "wouldn't", "youd": "you'd", "youre": "you're", "youve": "you've",
        }
        _vqa_number_words = {
            "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        }
        _vqa_articles = {"a", "an", "the"}
        _vqa_punct = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\',
                      '_', '-', '>', '<', '@', '`', ',', '?', '!', '.', ':']

        def normalize(s):
            """Official VQA-style answer normalization."""
            s = s.lower().strip()
            # Remove punctuation entirely (rather than spacing it out),
            # so "brown." normalizes to "brown", not "brown .".
            for p in _vqa_punct:
                s = s.replace(p, ' ')
            s = re.sub(r"\s+", " ", s).strip()

            words = []
            for word in s.split():
                word = _vqa_number_words.get(word, word)
                if word in _vqa_articles:
                    continue
                word = _vqa_contractions.get(word, word)
                words.append(word)
            return ' '.join(words).strip()

        def compute_vqa_metrics(p: EvalPrediction):
            pred_ids = np.asarray(p.predictions, dtype=np.int64)
            labels = np.asarray(p.label_ids, dtype=np.int64)

            vocab_size = processor.tokenizer.vocab_size
            pad_token_id = processor.tokenizer.pad_token_id

            pred_ids = np.clip(
                pred_ids,
                0,
                vocab_size - 1
            )

            try:
                pred_texts_raw = processor.batch_decode(
                    pred_ids,
                    skip_special_tokens=True
                )

                # IMPORTANT:
                # Labels contain -100 for all prompt tokens.
                # Decode ONLY the answer tokens.
                label_texts = []

                for label_row in labels:
                    valid_ids = label_row[
                        label_row != -100
                    ]

                    if valid_ids.size == 0:
                        label_texts.append("")
                        continue

                    valid_ids = np.clip(
                        valid_ids,
                        0,
                        vocab_size - 1
                    )

                    decoded = processor.tokenizer.decode(
                        valid_ids.tolist(),
                        skip_special_tokens=True
                    )

                    label_texts.append(decoded)

            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {
                    "rouge_l": 0.0,
                    "meteor": 0.0,
                    "bertscore_f1": 0.0,
                    "moverscore": 0.0,
                    "accuracy": 0.0
                }
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0, "accuracy": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            valid_preds = []
            valid_labels = []
            for p, l in zip(pred_norm, label_norm):
                if p and l and len(p.split()) > 0 and len(l.split()) > 0:
                    valid_preds.append(p)
                    valid_labels.append(l)

            # ROUGE-L
            rouge_ls = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    rouge_ls.append(scorer.score(l, p)['rougeL'].fmeasure)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR
            meteors = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    meteors.append(meteor_score([l.split()], p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore
            if valid_preds and valid_labels:
                try:
                    P, R, F1 = bert_score(
                        valid_preds,
                        valid_labels,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    bertscore_f1 = F1.mean().item()
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            raw_answers_list = eval_answers_buffer.get("answers", [])

            if len(raw_answers_list) != len(pred_norm):
                print(
                    f"[Metrics] WARNING: buffered answers ({len(raw_answers_list)}) "
                    f"!= predictions ({len(pred_norm)}); falling back to "
                    f"normalized exact-match accuracy against the single "
                    f"training answer instead of the official VQA formula."
                )
                matches = [1.0 if p == l else 0.0 for p, l in zip(pred_norm, label_norm) if l]
                accuracy = float(np.mean(matches)) if matches else 0.0
            else:
                per_question_accs = []
                for pred, raw_answers in zip(pred_norm, raw_answers_list):
                    norm_answers = [normalize(a) for a in raw_answers if a]
                    n = len(norm_answers)
                    if n == 0:
                        continue
                    if n == 1:
                        matches = sum(1 for a in norm_answers if a == pred)
                        per_question_accs.append(min(matches / 3.0, 1.0))
                        continue
                    leave_one_out_accs = []
                    for i in range(n):
                        others = norm_answers[:i] + norm_answers[i + 1:]
                        matches = sum(1 for a in others if a == pred)
                        leave_one_out_accs.append(min(matches / 3.0, 1.0))
                    per_question_accs.append(float(np.mean(leave_one_out_accs)))
                accuracy = float(np.mean(per_question_accs)) if per_question_accs else 0.0

            # MoverScore
            moverscore_val = 0.0
            if moverscore_available and valid_preds and valid_labels:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(valid_labels)
                    idf_dict_hyp = get_idf_dict(valid_preds)

                    scores = word_mover_score(
                        valid_labels,
                        valid_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        moverscore_val = float(np.mean(scores))
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc() # <--- Added this line

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
                "accuracy": accuracy,
            }

        return compute_vqa_metrics

    # Custom Trainer with VRAM clearing before evaluation
    class LoRATrainer(Seq2SeqTrainer):
        """Seq2SeqTrainer subclass that clears VRAM before evaluation."""

        def __init__(self, processor, eval_answers_buffer, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor
            # Mutable dict shared with make_compute_metrics(), used to pass
            # the raw per-question human answers (popped out of each batch
            # below) through to compute_metrics for the official VQA
            # soft-accuracy formula, since EvalPrediction only carries
            # decoded predictions/labels.
            self.eval_answers_buffer = eval_answers_buffer

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.eval_answers_buffer["answers"] = []
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: dict,
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ):
            """Memory-efficient prediction step from LoRA."""
            has_labels = "labels" in inputs
            if has_labels:
                labels = inputs.pop("labels")
            else:
                labels = None

            vqa_answers_batch = inputs.pop("vqa_answers", None)
            if vqa_answers_batch is not None:
                self.eval_answers_buffer["answers"].extend(vqa_answers_batch)

            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]

            answer_lengths = inputs.pop("answer_length", None)

            # ---------------------------------------------------------
            # Evaluation generation must use PROMPT ONLY.
            #
            # The training dataset contains:
            #   prompt + answer
            #
            # Remove the answer portion before calling generate().
            # ---------------------------------------------------------

            if answer_lengths is not None:
                prompt_input_ids = []
                prompt_attention_masks = []

                for i in range(input_ids.shape[0]):
                    ans_len = int(answer_lengths[i].item())

                    if ans_len > 0:
                        prompt_input_ids.append(
                            input_ids[i, :-ans_len]
                        )
                        prompt_attention_masks.append(
                            attention_mask[i, :-ans_len]
                        )
                    else:
                        prompt_input_ids.append(input_ids[i])
                        prompt_attention_masks.append(
                            attention_mask[i]
                        )

                # Left padding is used by the tokenizer, so pad prompt
                # sequences back to a common length.
                prompt_input_ids = torch.nn.utils.rnn.pad_sequence(
                    prompt_input_ids,
                    batch_first=True,
                    padding_value=self.processor.tokenizer.pad_token_id
                )

                prompt_attention_masks = torch.nn.utils.rnn.pad_sequence(
                    prompt_attention_masks,
                    batch_first=True,
                    padding_value=0
                )
            else:
                input_ids = inputs["input_ids"]
                input_len = input_ids.shape[1]

                with torch.no_grad():
                    if prediction_loss_only:
                        outputs = model(**inputs)
                        loss = outputs.loss if has_labels else None
                        return (loss, None, None)

                    generated_tokens = model.generate(
                        **inputs,
                        max_new_tokens=512,
                        do_sample=False,
                        pad_token_id=self.processor.tokenizer.pad_token_id,
                        eos_token_id=self.processor.tokenizer.eos_token_id,
                    )

                    generated_tokens = generated_tokens[:, input_len:]
                # Keep only newly generated tokens.


                # Slice to generated part only (answer)


                # Loss (full inputs for compute_loss)
                loss = None
                if has_labels:
                    inputs_copy = inputs.copy()
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            # Labels processing
            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels

    eval_answers_buffer = {"answers": []}
    tracker = ReproducibilityTracker()

    trainer = LoRATrainer(
        processor=processor,
        eval_answers_buffer=eval_answers_buffer,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor, eval_answers_buffer),
        callbacks=[
            ReproducibilityCallback(tracker),
        ],
    )

    print("\n" + "="*80)
    print("STARTING LORA-ONLY TRAINING FOR LLAMA 3.2 11B VISION MODEL")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        test_results = trainer.evaluate(
            eval_dataset=tokenized_test_dataset
        )

        repro_metrics = tracker.summary()
        total_steps = trainer.state.global_step

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")
        print(f"Test Accuracy      : {test_results['eval_accuracy']:.4f}")

        print("\n" + "="*80)
        print("REPRODUCIBILITY / EFFICIENCY METRICS")
        print("="*80)

        print(
            f"Training wall-clock : "
            f"{repro_metrics['training_time_hours']:.4f} hours"
        )

        print(
            f"Optimizer steps     : "
            f"{total_steps}"
        )

        print(
            f"Mean step latency   : "
            f"{repro_metrics['mean_step_ms']:.2f} ms"
        )

        print(
            f"P99 step latency    : "
            f"{repro_metrics['p99_step_ms']:.2f} ms"
        )

        print(
            f"Peak GPU allocated  : "
            f"{repro_metrics['peak_gpu_memory_allocated_gb']:.2f} GB"
        )

        print(
            f"Peak GPU reserved   : "
            f"{repro_metrics['peak_gpu_memory_reserved_gb']:.2f} GB"
        )
        final_results = {
            "test_metrics": test_results,
            "reproducibility_metrics": {
                "training_wall_clock_hours":
                    repro_metrics["training_time_hours"],

                "training_wall_clock_minutes":
                    repro_metrics["training_time_min"],

                "optimizer_steps":
                    total_steps,

                "mean_step_ms":
                    repro_metrics["mean_step_ms"],

                "p99_step_ms":
                    repro_metrics["p99_step_ms"],

                "peak_gpu_memory_allocated_gb":
                    repro_metrics[
                        "peak_gpu_memory_allocated_gb"
                    ],

                "peak_gpu_memory_reserved_gb":
                    repro_metrics[
                        "peak_gpu_memory_reserved_gb"
                    ],
            }
        }

        results_path = "LoRA_llama8_seed42_VQA.json"

        with open(results_path, "w") as f:
            json.dump(final_results, f, indent=2)

        print(f"\nResults saved to: {results_path}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = "./final_vqa_lora_llama_model"
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            model.save_pretrained("./interrupted_checkpoint_llama")
            processor.save_pretrained("./interrupted_checkpoint_llama")
            print("Checkpoint saved to ./interrupted_checkpoint_llama")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()


Loading processor and model for Lora Llama Vision...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

✓ Model has no meta parameters
✓ Model device: cuda:0
Instrumented 120 vision modules + 120 language modules
Total LoRA modules: 240
Trainable params: 4,177,920
Total params: 10,674,398,755
Percentage: 0.039%
Streaming VQAv2 dataset (no full download)...


Repo card metadata block was not found. Setting CardData to empty.
Parameter 'function'=<function main.<locals>.tokenize_vqa at 0x7f74352504a0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


... (33 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MoverScore] Score: 0.5032


Training Loss,Validation Loss,Epoch,Rouge L,Meteor,Bertscore F1,Moverscore,Accuracy
No log,32.118492,3,0.270833,0.319320,0.467834,0.503188,0.000000


Test Loss          : 32.11849
Test ROUGE-L       : 0.2708
Test METEOR        : 0.3193
Test BERTscore F1  : 0.4678
Test MoverScore    : 0.5032
Test Accuracy      : 0.0000

REPRODUCIBILITY / EFFICIENCY METRICS
Training wall-clock : 0.0623 hours
Optimizer steps     : 24
Mean step latency   : 4967.46 ms
P99 step latency    : 5436.43 ms
Peak GPU allocated  : 32.10 GB
Peak GPU reserved   : 32.72 GB

Results saved to: LoRA_llama11B_seed41_VQA_results.json

Saving final model...
Model saved to ./final_vqa_lora_llama_model

TRAINING COMPLETED SUCCESSFULLY


Cleaning up...
Cleanup complete
